# Update Water electric demand trajectory only

Takes the fully recalibrated input (`sisepuede_raw_inputs_oilgas_fgtv_recalibrated.csv`) and **only** rewrites the year-by-year scalar of `rubber_and_leather` (renamed as **"Water"** in this analysis) so that its INEN demand follows a new BAU trajectory. Every other input variable is preserved exactly as in the source CSV.

## Why

The source CSV was originally calibrated against an older Water-Energy model curve (16.84 PJ at 2020 → 49.54 PJ at 2050). The updated curve below is lower across the horizon (13.84 PJ at 2020 → 29.29 PJ at 2050) and we want to refresh the input **without** redoing the rest of the recalibration chain (electricity, INEN sectoral targets, oil & gas, FGTV).

## Assumptions on the source CSV

Verified at notebook-load time (cell 1):
- `frac_inen_energy_rubber_and_leather_electricity = 1.0` and other 12 fractions = 0 across all years (also for `recycled_rubber_and_leather`).
- `prodinit_ippu_rubber_and_leather_tonne` is non-zero (production-rescue boost was already applied upstream).
- `scalar_inen_energy_demand_rubber_and_leather` is present and currently encodes the old trajectory.

These conditions hold in `sisepuede_raw_inputs_oilgas_fgtv_recalibrated.csv`, so the notebook does **not** re-apply them.

## Method

Run the **INEN model only** (no NemoMod) on the source CSV to obtain the current per-year demand `D_current(t)` for `rubber_and_leather`. Then overwrite the scalar column with

$$\text{scalar}_{\text{new}}(t) \;=\; \text{target}_{\text{new}}(t)\,\cdot\,\frac{\text{scalar}_{\text{current}}(t)}{D_{\text{current}}(t)}$$

Years outside the trajectory's domain (2015–2019) are flat-extrapolated at the 2020 value, matching the convention used in `recalibrate_inen_energy_demand.ipynb`.

## Parameters

In [1]:
BASE_INPUT_CSV = "../../input_data/sisepuede_raw_inputs_oilgas_fgtv_recalibrated.csv"
OUT_INPUT_CSV  = "../../input_data/sisepuede_raw_inputs_oilgas_fgtv_water_updated.csv"

WATER_SECTOR = "rubber_and_leather"
BASE_YEAR    = 2015
REGION       = "libya"

# Updated Water BAU trajectory (PJ/yr). Replaces the older curve embedded in the source CSV.
WATER_BAU_TRAJECTORY_PJ = {
    2020: 13.84, 2021: 14.30, 2022: 14.76, 2023: 15.23,
    2024: 15.69, 2025: 16.15, 2026: 16.62, 2027: 17.16,
    2028: 17.71, 2029: 18.26, 2030: 18.81, 2031: 19.37,
    2032: 19.92, 2033: 20.43, 2034: 20.95, 2035: 21.48,
    2036: 22.00, 2037: 22.53, 2038: 23.06, 2039: 23.59,
    2040: 24.13, 2041: 24.67, 2042: 25.19, 2043: 25.69,
    2044: 26.20, 2045: 26.70, 2046: 27.21, 2047: 27.73,
    2048: 28.25, 2049: 28.76, 2050: 29.29,
}

In [2]:
import os, sys, pathlib, warnings, logging
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

HERE = pathlib.Path(os.getcwd())
sys.path.insert(0, str(HERE.parent))
from utils.logger_utils import setup_clean_logger, mute_external_loggers
logger = setup_clean_logger('water_update', logging.INFO)
mute_external_loggers(['sisepuede'])

## 1. Sanity-check the source CSV

Confirm that the source CSV already carries the upstream calibration that this notebook relies on (electricity-only fuel mix and a present scalar column). If any check fails, fix the upstream notebook (`recalibrate_inen_energy_demand.ipynb`) instead of patching here.

In [3]:
df_in = pd.read_csv(BASE_INPUT_CSV)
scalar_col = f'scalar_inen_energy_demand_{WATER_SECTOR}'
assert scalar_col in df_in.columns, f'Missing column {scalar_col} in source CSV'
assert 'year' in df_in.columns, 'Source CSV is missing the year column'

for variant in [WATER_SECTOR, f'recycled_{WATER_SECTOR}']:
    prefix = f'frac_inen_energy_{variant}_'
    frac_cols = [c for c in df_in.columns if c.startswith(prefix)]
    if not frac_cols:
        continue
    elec_col = f'{prefix}electricity'
    others   = [c for c in frac_cols if c != elec_col]
    assert (df_in[elec_col] == 1.0).all(), f'{elec_col} != 1.0 in some year'
    assert (df_in[others].sum(axis=1) == 0.0).all(), f'{variant} non-electricity fuel > 0 in some year'
    print(f'  OK {variant}: electricity=1, others=0 in every year ({len(frac_cols)} cols)')

scalar_base = df_in.set_index('year')[scalar_col]
print(f'\nCurrent scalar @2020/2023/2050: '
      f'{scalar_base.loc[2020]:.4f} / {scalar_base.loc[2023]:.4f} / {scalar_base.loc[2050]:.4f}')

  OK rubber_and_leather: electricity=1, others=0 in every year (13 cols)
  OK recycled_rubber_and_leather: electricity=1, others=0 in every year (13 cols)

Current scalar @2020/2023/2050: 0.0400 / 0.0387 / 0.0511


## 2. Measure current Water INEN demand on the source CSV

Run the EnergyConsumption (INEN) model only to obtain `energy_demand_inen_rubber_and_leather` per year. With the source CSV's existing scalar this curve should reproduce the **old** Water trajectory.

In [4]:
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
from ssp_transformations_handler.GeneralUtils import GeneralUtils

_EXAMPLES = sxl.SISEPUEDEExamples()

def run_inen_only(csv_path: str, y0: int = BASE_YEAR, y1: int = 2050) -> pd.DataFrame:
    """Run only the EnergyConsumption (INEN) model, no NemoMod."""
    df_raw = pd.read_csv(csv_path)
    fs = sfs.SISEPUEDEFileStructure(initialize_directories=False)
    key_tp, key_year = fs.model_attributes.dim_time_period, fs.model_attributes.field_dim_year
    years = np.arange(y0, y1 + 1).astype(int)
    att_tp = att.AttributeTable(
        pd.DataFrame({key_tp: range(len(years)), key_year: years}), key_tp,
    )
    fs.model_attributes.update_dimensional_attribute_table(att_tp)
    matt = fs.model_attributes
    models = sm.SISEPUEDEModels(
        matt, allow_electricity_run=False,
        fp_julia=fs.dir_jl, fp_nemomod_reference_files=fs.dir_ref_nemo,
        initialize_julia=False,
    )
    df_ex = _EXAMPLES('input_data_frame')
    df = GeneralUtils().add_missing_cols(df_ex, df_raw.copy())
    df['region'] = REGION
    df = df[df['year'] <= y1]
    df_out = models.project(df, include_electricity_in_energy=False)
    df_out['year'] = df_out['time_period'].apply(lambda t: y0 + int(t))
    return df_out

df_meas = run_inen_only(BASE_INPUT_CSV)
demand_col = f'energy_demand_inen_{WATER_SECTOR}'
d_current  = df_meas.set_index('year')[demand_col]
print(f'Current Water INEN demand on source CSV:')
for y in [2020, 2023, 2030, 2050]:
    print(f'  {y}: {float(d_current.loc[y]):.2f} PJ')

No missing columns to add.
Current Water INEN demand on source CSV:
  2020: 16.84 PJ
  2023: 21.40 PJ
  2030: 30.85 PJ
  2050: 49.54 PJ


## 3. Compute the new scalar and write the updated CSV

For every year `t` covered by the model (2015–2050) we set

$$\text{scalar}_{\text{new}}(t) \;=\; \text{target}_{\text{new}}(t)\,\cdot\,\frac{\text{scalar}_{\text{current}}(t)}{D_{\text{current}}(t)}$$

with flat extrapolation for years outside `WATER_BAU_TRAJECTORY_PJ` (2015–2019 → 2020 value). Only the column `scalar_inen_energy_demand_rubber_and_leather` is touched; the rest of the CSV is written back unchanged.

In [5]:
def _traj_value(traj: dict, year: int) -> float:
    """Flat extrapolation outside [min(years), max(years)]."""
    ymin, ymax = min(traj), max(traj)
    if year <= ymin:
        return traj[ymin]
    if year >= ymax:
        return traj[ymax]
    return traj[year]

scalar_new = {}
for y in df_in['year'].astype(int):
    target_y = _traj_value(WATER_BAU_TRAJECTORY_PJ, int(y))
    d_y      = float(d_current.loc[y]) if y in d_current.index else 0.0
    s_b      = float(scalar_base.loc[y])
    if d_y <= 0.0:
        raise RuntimeError(
            f'{WATER_SECTOR}: source-CSV demand is {d_y:.6f} PJ at {y} — '
            f'cannot reshape trajectory via scalar (need > 0).'
        )
    scalar_new[int(y)] = target_y * s_b / d_y

df_in[scalar_col] = df_in['year'].astype(int).map(scalar_new).astype(float)

os.makedirs(os.path.dirname(OUT_INPUT_CSV), exist_ok=True)
df_in.to_csv(OUT_INPUT_CSV, index=False)

print(f'New scalar range: {min(scalar_new.values()):.4f} .. {max(scalar_new.values()):.4f}')
print(f'Wrote updated CSV to: {OUT_INPUT_CSV}')

New scalar range: 0.0195 .. 0.0329
Wrote updated CSV to: ../../input_data/sisepuede_raw_inputs_oilgas_fgtv_water_updated.csv


## 4. Verify

Re-run the INEN model on the output CSV and confirm that the Water demand matches the new trajectory within ±1% across every year of its domain. Also report endpoint values for context.

In [6]:
df_check = run_inen_only(OUT_INPUT_CSV)
demand_new = df_check.set_index('year')[demand_col]

rows = []
for y in sorted(WATER_BAU_TRAJECTORY_PJ):
    target_y = WATER_BAU_TRAJECTORY_PJ[y]
    new_y    = float(demand_new.loc[y])
    dev_pct  = (new_y - target_y) / target_y * 100 if target_y > 0 else 0.0
    rows.append({'year': y, 'target_PJ': round(target_y, 4),
                 'new_PJ': round(new_y, 4), 'dev_%': round(dev_pct, 4)})
verify_df = pd.DataFrame(rows)

print('Endpoints:')
print(verify_df.iloc[[0, -1]].to_string(index=False))
print('\nWorst 5 deviations:')
worst = verify_df.reindex(verify_df['dev_%'].abs().sort_values(ascending=False).index).head(5)
print(worst.to_string(index=False))

max_dev = verify_df['dev_%'].abs().max()
assert max_dev < 1.0, f'Some year deviated >1% from the new trajectory (max |dev| = {max_dev:.3f}%)'
print(f'\nOK - Water trajectory matches new BAU within +/-1% (max |dev| = {max_dev:.3f}%).')

# 2015-2019 sanity check (flat extrapolation at the 2020 value)
extrap_rows = []
for y in [2015, 2016, 2017, 2018, 2019]:
    if y not in demand_new.index:
        continue
    extrap_rows.append({'year': y, 'extrap_target_PJ': WATER_BAU_TRAJECTORY_PJ[2020],
                        'new_PJ': round(float(demand_new.loc[y]), 4)})
if extrap_rows:
    print('\nFlat-extrapolated years (2015-2019):')
    print(pd.DataFrame(extrap_rows).to_string(index=False))

No missing columns to add.
Endpoints:
 year  target_PJ  new_PJ  dev_%
 2020      13.84   13.84   -0.0
 2050      29.29   29.29   -0.0

Worst 5 deviations:
 year  target_PJ  new_PJ  dev_%
 2020      13.84   13.84   -0.0
 2036      22.00   22.00   -0.0
 2049      28.76   28.76   -0.0
 2048      28.25   28.25   -0.0
 2047      27.73   27.73    0.0

OK - Water trajectory matches new BAU within +/-1% (max |dev| = 0.000%).

Flat-extrapolated years (2015-2019):
 year  extrap_target_PJ  new_PJ
 2015             13.84   13.84
 2016             13.84   13.84
 2017             13.84   13.84
 2018             13.84   13.84
 2019             13.84   13.84
